# 🪑 Hunyuan3D Chair Studio (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Solamente0/hunyuan3d-google-colab/blob/main/Hunyuan3D_Chairs.ipynb)

**Runtime:** `Runtime → Change runtime type → A100 GPU` + **High-RAM**.

1. **Setup**: pulls the latest code from GitHub and checks the GPU
2. **Install**: about 10–20 minutes on a fresh runtime, then skipped
3. **Chair Studio**: upload photos → realistic textured 3D chairs
4. Optional: Tencent's official UI · logs · stop

All the logic lives in the GitHub repo, so pushing a change there is enough. Rerun cell 1 (or 3) to pick it up.

In [ ]:
#@title 1️⃣ Setup: get the latest code + check GPU
REPO_URL = "https://github.com/Solamente0/hunyuan3d-google-colab"  #@param {type:"string"}
CACHE_WEIGHTS_IN_DRIVE = False  #@param {type:"boolean"}
#@markdown ↑ Keeps the ~20 GB of weights in Google Drive so new sessions skip the download.
TOOLS_DIR = "/content/hunyuan3d-google-colab"

import os, subprocess
if os.path.isdir(f"{TOOLS_DIR}/.git"):
    subprocess.run(["git", "-C", TOOLS_DIR, "pull", "-q", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, TOOLS_DIR], check=True)
print("Code version:", subprocess.run(["git", "-C", TOOLS_DIR, "log", "-1", "--format=%h  %s  (%cr)"],
                                      capture_output=True, text=True).stdout.strip())
%run -i {TOOLS_DIR}/setup_env.py


In [ ]:
#@title 2️⃣ Install Hunyuan3D 2.1 (first time per runtime)
!bash {TOOLS_DIR}/scripts/install.sh


## 🪑 Getting the most realistic chair textures

Chair Studio (cell 3) already uses the highest quality settings and runs as fast as your GPU allows: weights are pre-downloaded, the mesh is extracted with FlashVDM (much faster than the standard decoder at the same resolution), and on a 40 GB+ GPU shape generation and texturing run **at the same time** — on 80 GB with two texture workers. The **photo** is now what decides the quality:

- **One chair per photo, a 3/4 front view** (slightly from above, so the seat, front and one side are all visible). The texture model copies colors and materials from this photo onto the 3D shape, so anything hidden in the photo has to be guessed.
- **Soft, even light** — no harsh shadows or strong reflections; the model "de-lights" the photo, but baked-in shadows can still end up in the texture.
- **High resolution, sharp, chair filling most of the frame.** Fabric, wood grain and leather detail come from here.
- **Plain background**, or better: upload a **PNG with the background already removed** (transparent). Chairs have gaps between legs, rungs and slats where automatic background removal sometimes leaves bits of floor. Transparent PNGs are used as-is.
- **Already have 3D models of the chairs?** Upload each model with the same file name as its photo (`armchair.jpg` + `armchair.obj`). Chair Studio then skips shape generation and only paints your model, keeping its geometry. The photo should show the chair from the front, matching the model's orientation.

Every output folder also contains `input_clean.png` (the photo after background removal). If a texture looks wrong, check that file first.

In [ ]:
#@title 3️⃣ Chair Studio: upload → generate → preview → download
#@markdown Opens the app below plus a public **gradio.live** link. Drag in all your chair photos at once;
#@markdown to texture your own 3D models, add each one with the same file name as its photo (`armchair.jpg` + `armchair.obj`).
!git -C {TOOLS_DIR} pull -q --ff-only
%run -i {TOOLS_DIR}/chair_studio.py


In [ ]:
#@title 4️⃣ (Optional) Tencent's official web UI, one image at a time
PUBLIC_LINK = False  #@param {type:"boolean"}
FORCE_LOW_VRAM_MODE = False  #@param {type:"boolean"}
%run -i {TOOLS_DIR}/official_ui.py


In [ ]:
#@title 5️⃣ Official UI logs (run anytime)
LINES = 40  #@param {type:"integer"}
!tail -n {LINES} /content/hunyuan3d.log
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


In [ ]:
#@title 6️⃣ Stop the server (frees the GPU)
!pkill -f gradio_app.py; pkill -f run_chairs.py; pkill -f "cloudflared tunnel"; echo "Stopped."
